# Phase 1C: Acceptance Rate Measurement
**Purpose:** Measure per-iteration acceptance rates for linear SD and prompt lookup to enable theory-vs-practice gap analysis.

Uses same models as Phase 1 (Llama-3.2-3B target + Llama-3.2-1B draft).
Runs a smaller set of trials but captures acceptance statistics.


In [6]:
!pip install -q transformers accelerate bitsandbytes

In [7]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
import torch
import time
import json
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available()
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)")

GPU: Tesla T4 (15.6 GB)


In [9]:
# ============================================================
# Config — same as Phase 1
# ============================================================
TARGET_MODEL = "meta-llama/Llama-3.2-3B"
DRAFT_MODEL  = "meta-llama/Llama-3.2-1B"
MAX_NEW_TOKENS = 128

PROMPTS = [
    "The history of artificial intelligence began in the 1950s when researchers first",
    "In computer science, a hash table is a data structure that implements an associative",
    "The process of photosynthesis converts light energy into chemical energy through",
    'def fibonacci(n):\n    """Calculate the nth Fibonacci number."""\n    if n <= 1:\n        return n\n',
    "# Python implementation of binary search\ndef binary_search(arr, target):\n",
    "Summarize the following: Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data. Machine learning algorithms use historical data as input to predict new output values. Machine learning is",
    "Write a detailed explanation of how TCP/IP networking works, starting from",
    "Explain the difference between a stack and a queue data structure. A stack",
    "The transformer architecture was introduced in the paper Attention is All You Need by Vaswani et al. in 2017. It replaced recurrent neural networks with self-attention mechanisms, enabling parallel processing of sequences. The key innovation was the multi-head attention mechanism, which allows the model to attend to different parts of the input simultaneously. Since then, transformers have become the foundation for",
    "Large language models are trained on massive datasets of text from the internet. During training, the model learns to predict the next token in a sequence given all previous tokens. This autoregressive training objective means that at inference time, the model generates text one token at a time, each conditioned on all previously generated tokens. This sequential nature creates a fundamental bottleneck because",
]

PROMPT_LABELS = [
    "general_1", "general_2", "general_3",
    "code_1", "code_2", "summarization",
    "instruction_1", "instruction_2",
    "long_ctx_1", "long_ctx_2",
]

print(f"Target: {TARGET_MODEL}")
print(f"Draft:  {DRAFT_MODEL}")

Target: meta-llama/Llama-3.2-3B
Draft:  meta-llama/Llama-3.2-1B


In [11]:
# Load models
print("Loading target...")
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, torch_dtype=torch.float16, device_map="auto",
)
target_model.eval()

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading draft...")
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL, torch_dtype=torch.float16, device_map="auto",
)
draft_model.eval()

alloc = torch.cuda.memory_allocated() / 1e9
print(f"Total GPU: {alloc:.2f} GB")
print("Ready.")

Loading target...


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Total GPU: 8.90 GB
Ready.


## Acceptance Rate Measurement

**Method:** We run speculative decoding with `return_dict_in_generate=True` and `output_scores=True`, then compare the number of tokens generated per forward pass of the target model.

HuggingFace's assisted generation internally tracks how many draft tokens are accepted each iteration. We can measure this by comparing the total generated tokens to the number of target model forward passes (iterations). The ratio gives us mean accepted tokens per iteration, from which we can derive acceptance rate.

**Alternative approach:** Generate with SD and without SD using greedy decoding (deterministic). The outputs must be identical (exact decoding guarantee). We count target forward passes by measuring how many "iterations" SD takes vs how many tokens it produces.


In [12]:
# ============================================================
# Measure acceptance rate using manual speculative decoding loop
# ============================================================
#
# Strategy: Run the draft model for k steps, then verify with target.
# Count how many tokens match. This gives us alpha directly.
#
# We implement a simple manual spec decode loop so we can
# instrument every iteration.

def measure_acceptance_rate(target_model, draft_model, tokenizer, prompt,
                            k=5, max_new_tokens=128):
    """
    Manual speculative decoding loop that records per-iteration acceptance.
    Returns dict with acceptance statistics.
    """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(target_model.device)

    total_generated = 0
    total_draft_tokens = 0
    total_accepted = 0
    num_iterations = 0
    per_iter_accepted = []

    with torch.inference_mode():
        while total_generated < max_new_tokens:
            # --- Draft phase: generate k tokens with draft model ---
            draft_input = input_ids.clone()
            draft_tokens = []

            for _ in range(k):
                draft_out = draft_model(draft_input)
                next_token = draft_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                draft_tokens.append(next_token.item())
                draft_input = torch.cat([draft_input, next_token], dim=-1)

            # --- Verify phase: run target model on input + all draft tokens ---
            verify_input = torch.cat([
                input_ids,
                torch.tensor([draft_tokens], device=input_ids.device)
            ], dim=-1)

            target_out = target_model(verify_input)
            # target_out.logits shape: [1, seq_len, vocab_size]
            # We need logits at positions corresponding to draft tokens

            # The target model's prediction for position i is at logits[:, i, :]
            # For draft token at position j (0-indexed within draft),
            # the target's prediction is at logits[:, prompt_len + j - 1, :]
            prompt_len = input_ids.shape[1]

            accepted = 0
            for j in range(k):
                target_pred = target_out.logits[:, prompt_len + j - 1, :].argmax(dim=-1).item()
                if target_pred == draft_tokens[j]:
                    accepted += 1
                else:
                    break  # stop at first rejection

            # After accepted tokens, take the target model's next token
            # (either the correction token or the token after all accepted)
            bonus_pos = prompt_len + accepted - 1
            bonus_token = target_out.logits[:, bonus_pos, :].argmax(dim=-1, keepdim=True)

            # Update input_ids: keep prompt + accepted draft tokens + bonus token
            if accepted > 0:
                accepted_tensor = torch.tensor([draft_tokens[:accepted]], device=input_ids.device)
                input_ids = torch.cat([input_ids, accepted_tensor, bonus_token], dim=-1)
            else:
                input_ids = torch.cat([input_ids, bonus_token], dim=-1)

            tokens_this_iter = accepted + 1  # accepted drafts + 1 bonus
            total_generated += tokens_this_iter
            total_draft_tokens += k
            total_accepted += accepted
            num_iterations += 1
            per_iter_accepted.append(accepted)

            # Check for EOS
            if bonus_token.item() == tokenizer.eos_token_id:
                break
            if accepted > 0 and tokenizer.eos_token_id in draft_tokens[:accepted]:
                break

    alpha = total_accepted / total_draft_tokens if total_draft_tokens > 0 else 0
    mean_accepted_per_iter = np.mean(per_iter_accepted) if per_iter_accepted else 0

    return {
        "total_generated": total_generated,
        "total_draft_tokens": total_draft_tokens,
        "total_accepted": total_accepted,
        "num_iterations": num_iterations,
        "acceptance_rate": round(alpha, 4),
        "mean_accepted_per_iter": round(float(mean_accepted_per_iter), 2),
        "tokens_per_target_call": round(total_generated / num_iterations, 2) if num_iterations > 0 else 0,
        "per_iter_accepted": per_iter_accepted,
    }

print("Acceptance rate measurement function ready.")

Acceptance rate measurement function ready.


## Run Acceptance Rate Experiments

In [13]:
# ============================================================
# Measure acceptance rates across all prompts and k values
# ============================================================
K_VALUES = [3, 5, 7, 10]
all_acceptance_results = []

print("Measuring acceptance rates...")
print(f"{'Prompt':<16} ", end="")
for k in K_VALUES:
    print(f"{'k='+str(k):>10}", end="")
print()
print("-" * 60)

for pidx, prompt in enumerate(PROMPTS):
    label = PROMPT_LABELS[pidx]
    print(f"{label:<16} ", end="")

    for k in K_VALUES:
        result = measure_acceptance_rate(
            target_model, draft_model, tokenizer, prompt, k=k, max_new_tokens=MAX_NEW_TOKENS
        )
        result["prompt_idx"] = pidx
        result["prompt_label"] = label
        result["k"] = k
        all_acceptance_results.append(result)

        print(f"{result['acceptance_rate']:>10.3f}", end="")
    print()

# Save
with open("acceptance_rates.json", "w") as f:
    json.dump(all_acceptance_results, f, indent=2)
print(f"\nSaved {len(all_acceptance_results)} measurements to acceptance_rates.json")

Measuring acceptance rates...
Prompt                  k=3       k=5       k=7      k=10
------------------------------------------------------------
general_1             0.707     0.626     0.534     0.457
general_2             0.698     0.553     0.476     0.396
general_3             0.674     0.588     0.517     0.396
code_1                0.933     1.000     0.762     0.850
code_2                0.707     0.613     0.488     0.458
summarization         0.706     0.606     0.550     0.483
instruction_1         0.615     0.492     0.411     0.341
instruction_2         0.786     0.703     0.606     0.579
long_ctx_1            0.778     0.683     0.631     0.495
long_ctx_2            0.718     0.581     0.524     0.425

Saved 40 measurements to acceptance_rates.json


## Summary Statistics

In [14]:
# Aggregate by k
print("\n=== Acceptance Rate by k ===")
print(f"{'k':>4} {'Mean alpha':>12} {'Std':>8} {'Min':>8} {'Max':>8} {'Mean tok/iter':>14}")
print("-" * 60)

for k in K_VALUES:
    entries = [r for r in all_acceptance_results if r["k"] == k]
    alphas = [r["acceptance_rate"] for r in entries]
    tpi = [r["tokens_per_target_call"] for r in entries]
    print(f"{k:>4} {np.mean(alphas):>12.4f} {np.std(alphas):>8.4f} "
          f"{min(alphas):>8.4f} {max(alphas):>8.4f} {np.mean(tpi):>14.2f}")

# Aggregate by prompt category
print("\n=== Acceptance Rate by Prompt (k=5) ===")
print(f"{'Prompt':<16} {'alpha':>8} {'tok/iter':>10} {'iterations':>12}")
print("-" * 50)
for r in all_acceptance_results:
    if r["k"] == 5:
        print(f"{r['prompt_label']:<16} {r['acceptance_rate']:>8.4f} "
              f"{r['tokens_per_target_call']:>10.2f} {r['num_iterations']:>12}")


=== Acceptance Rate by k ===
   k   Mean alpha      Std      Min      Max  Mean tok/iter
------------------------------------------------------------
   3       0.7324   0.0811   0.6148   0.9333           3.20
   5       0.6444   0.1316   0.4919   1.0000           4.22
   7       0.5499   0.0923   0.4113   0.7619           4.85
  10       0.4881   0.1355   0.3414   0.8500           5.88

=== Acceptance Rate by Prompt (k=5) ===
Prompt              alpha   tok/iter   iterations
--------------------------------------------------
general_1          0.6258       4.13           31
general_2          0.5529       3.76           34
general_3          0.5879       3.94           33
code_1             1.0000       6.00            3
code_2             0.6125       4.06           32
summarization      0.6062       4.03           32
instruction_1      0.4919       3.46           37
instruction_2      0.7034       4.52           29
long_ctx_1         0.6828       4.41           29
long_ctx_2       

## Measure Cd and Ct Separately

In [15]:
# ============================================================
# Measure per-token cost for draft and target independently
# ============================================================

def measure_per_token_cost(model, tokenizer, prompt, max_new_tokens=64, num_runs=5):
    """Measure average per-token generation time for a model."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    # Warmup
    with torch.inference_mode():
        model.generate(input_ids, max_new_tokens=10, do_sample=False,
                       pad_token_id=tokenizer.pad_token_id)

    times = []
    for _ in range(num_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.inference_mode():
            out = model.generate(input_ids, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        gen_tokens = out.shape[1] - input_ids.shape[1]
        times.append((t1 - t0) / gen_tokens)

    return {
        "mean_ms_per_token": round(float(np.mean(times)) * 1000, 2),
        "std_ms_per_token": round(float(np.std(times)) * 1000, 2),
    }

# Measure on a representative prompt
test_prompt = PROMPTS[0]

print("Measuring per-token costs...")
print(f"Using prompt: '{test_prompt[:50]}...'")

target_cost = measure_per_token_cost(target_model, tokenizer, test_prompt)
draft_cost = measure_per_token_cost(draft_model, tokenizer, test_prompt)

Ct = target_cost["mean_ms_per_token"]
Cd = draft_cost["mean_ms_per_token"]

print(f"\nTarget (3B fp16): Ct = {Ct:.2f} ms/token")
print(f"Draft  (1B fp16): Cd = {Cd:.2f} ms/token")
print(f"Cost ratio Cd/Ct = {Cd/Ct:.4f}")
print(f"\nFor reference: parameter ratio = 1B/3B = 0.333")
print(f"Cost ratio vs param ratio tells us about compute efficiency scaling.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Measuring per-token costs...
Using prompt: 'The history of artificial intelligence began in th...'

Target (3B fp16): Ct = 37.63 ms/token
Draft  (1B fp16): Cd = 22.42 ms/token
Cost ratio Cd/Ct = 0.5958

For reference: parameter ratio = 1B/3B = 0.333
Cost ratio vs param ratio tells us about compute efficiency scaling.


## Theory vs Practice Comparison

In [16]:
# ============================================================
# Compute theoretical speedup from Equation 1 and compare
# to measured speedup from Phase 1
# ============================================================

# Measured values from Phase 1
PHASE1_SPEEDUPS = {3: 1.010, 5: 0.977, 7: 0.967, 10: 0.921}

print("=== Theory vs Practice (Linear SD, 1B/3B, greedy) ===")
print(f"Ct = {Ct:.2f} ms,  Cd = {Cd:.2f} ms,  Cd/Ct = {Cd/Ct:.3f}")
print()
print(f"{'k':>4} {'alpha':>8} {'S_predicted':>13} {'S_measured':>12} {'Gap':>8}")
print("-" * 50)

for k in K_VALUES:
    # Get mean alpha for this k
    entries = [r for r in all_acceptance_results if r["k"] == k]
    alpha = np.mean([r["acceptance_rate"] for r in entries])

    # Equation 1: S = (alpha * k * Ct) / (k * Cd + Ct)
    S_predicted = (alpha * k * Ct) / (k * Cd + Ct)
    S_measured = PHASE1_SPEEDUPS.get(k, 0)
    gap = S_measured - S_predicted

    print(f"{k:>4} {alpha:>8.4f} {S_predicted:>13.3f}x {S_measured:>11.3f}x {gap:>+8.3f}")

print()
print("Gap < 0 means measured is WORSE than theory predicts.")
print("This residual is C_sys (cache rewind, scheduling, synchronization).")

# Also compute: what alpha would be needed for speedup > 1.0?
print("\n=== Minimum alpha for speedup > 1.0 ===")
for k in K_VALUES:
    # S > 1.0 requires: alpha * k * Ct > k * Cd + Ct
    # alpha > (k * Cd + Ct) / (k * Ct)
    alpha_min = (k * Cd + Ct) / (k * Ct)
    print(f"  k={k}: alpha > {alpha_min:.4f}")

=== Theory vs Practice (Linear SD, 1B/3B, greedy) ===
Ct = 37.63 ms,  Cd = 22.42 ms,  Cd/Ct = 0.596

   k    alpha   S_predicted   S_measured      Gap
--------------------------------------------------
   3   0.7324         0.788x       1.010x   +0.222
   5   0.6444         0.810x       0.977x   +0.167
   7   0.5499         0.745x       0.967x   +0.222
  10   0.4881         0.701x       0.921x   +0.220

Gap < 0 means measured is WORSE than theory predicts.
This residual is C_sys (cache rewind, scheduling, synchronization).

=== Minimum alpha for speedup > 1.0 ===
  k=3: alpha > 0.9291
  k=5: alpha > 0.7958
  k=7: alpha > 0.7387
  k=10: alpha > 0.6958


In [17]:
# Save everything
results_summary = {
    "Ct_ms": Ct,
    "Cd_ms": Cd,
    "cost_ratio": round(Cd/Ct, 4),
    "acceptance_by_k": {},
    "acceptance_by_prompt": {},
}

for k in K_VALUES:
    entries = [r for r in all_acceptance_results if r["k"] == k]
    results_summary["acceptance_by_k"][str(k)] = {
        "mean_alpha": round(float(np.mean([r["acceptance_rate"] for r in entries])), 4),
        "std_alpha": round(float(np.std([r["acceptance_rate"] for r in entries])), 4),
    }

for r in all_acceptance_results:
    if r["k"] == 5:
        results_summary["acceptance_by_prompt"][r["prompt_label"]] = r["acceptance_rate"]

with open("phase1c_summary.json", "w") as f:
    json.dump(results_summary, f, indent=2)

print("All results saved.")
print("\nCopy for report:")
print(f"  Ct = {Ct:.2f} ms/token")
print(f"  Cd = {Cd:.2f} ms/token")
print(f"  Cd/Ct = {Cd/Ct:.4f}")
print(f"  Mean alpha (k=5): {results_summary['acceptance_by_k']['5']['mean_alpha']}")

All results saved.

Copy for report:
  Ct = 37.63 ms/token
  Cd = 22.42 ms/token
  Cd/Ct = 0.5958
  Mean alpha (k=5): 0.6444
